In [1]:
using LowLevelFEM

In [2]:
structured_rect_mesh()

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:fu)
Pv = Problem([mat], type=:VectorField, dim=2, field=:v, rhs_field=:fv)
PT = Problem([mat], type=:ScalarField, dim=2, field=:T, rhs_field=:q)

Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)

In [3]:
E = mat.E
ν = mat.ν
k = mat.k
α = mat.α
κ = mat.κ
ρ = mat.ρ
c = mat.c

T0 = 293

D = E / (1 + ν^2) * [1 ν 0; ν 1 0; 0 0 (1-ν)/2]

3×3 Matrix{Float64}:
     1.83486e5  55045.9            0.0
 55045.9            1.83486e5      0.0
     0.0            0.0        64220.2

In [4]:
Mu = ∫(Pu ⋅ Pu)
Mv = ∫(Pv ⋅ Pv * ρ)
MT = ∫(PT ⋅ PT * (c * ρ))

sparse([1, 5, 40, 41, 2, 13, 14, 113, 3, 22  …  121, 3, 21, 22, 23, 24, 111, 112, 120, 121], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  120, 121, 121, 121, 121, 121, 121, 121, 121, 121], [0.0036633333333333314, 0.0018316666666666663, 0.0018316666666666663, 0.0009158333333333332, 0.0036633333333333322, 0.0018316666666666666, 0.0018316666666666666, 0.0009158333333333332, 0.0036633333333333305, 0.0018316666666666663  …  0.003663333333333332, 0.0009158333333333332, 0.0009158333333333336, 0.0036633333333333327, 0.0036633333333333327, 0.0009158333333333332, 0.0009158333333333325, 0.0036633333333333322, 0.003663333333333332, 0.014653333333333326], 121, 121)

In [5]:
Muv = ∫(Pu ⋅ Pv * 0)
MuT = ∫(Pu ⋅ [0; 0;;] ⋅ PT)
Mvu = ∫(Pv ⋅ Pu * 0)
MvT = ∫(Pv ⋅ [0; 0;;] ⋅ PT)
MTu = ∫(PT ⋅ [0 0] ⋅ Pu)
MTv = ∫(PT ⋅ [0 0] ⋅ Pv)

sparse(Int64[], Int64[], Float64[], 121, 242)

In [6]:
αI = [1; 1; 0;;] * α

3×1 Matrix{Float64}:
 1.2e-5
 1.2e-5
 0.0

In [7]:
Kuv = ∫(Pu ⋅ Pv)

Kvu = ∫(SymGrad(Pv) ⋅ D ⋅ SymGrad(Pu))
KvT = ∫(SymGrad(Pv) ⋅ D ⋅ αI ⋅ PT)

KTu = ∫(PT ⋅ Div(Pu) * (κ * T0 * 1000000))
KTT = ∫(Grad(PT) ⋅ k ⋅ Grad(PT))

sparse([1, 5, 40, 41, 2, 13, 14, 113, 3, 22  …  121, 3, 21, 22, 23, 24, 111, 112, 120, 121], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  120, 121, 121, 121, 121, 121, 121, 121, 121, 121], [29.999999999999993, -7.499999999999996, -7.500000000000001, -15.0, 29.999999999999996, -7.500000000000001, -7.499999999999997, -15.0, 30.0, -7.500000000000001  …  -15.00000000000001, -15.000000000000004, -15.00000000000001, -14.999999999999993, -15.0, -14.999999999999998, -14.999999999999993, -14.99999999999999, -15.00000000000001, 120.0], 121, 121)

In [8]:
Kuu = ∫(Pu ⋅ Pu * 0)
KuT = ∫(Pu ⋅ [0; 0;;] ⋅ PT)

Kvv = ∫(Pv ⋅ Pv * 0)

KTv = ∫(PT ⋅ [0 0] ⋅ Pv)

sparse(Int64[], Int64[], Float64[], 121, 242)

In [9]:
K = SystemMatrix([Kuu Kuv KuT; Kvu Kvv KvT; KTu KTv KTT])

K[:, :]

605×605 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10329 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡻⣌⠐⠒⠂⠃⠶⠰⠠⠀⠄⡄⡄⢀⢈⠐⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠇⠈⠳⣄⠀⠀⠀⢀⢀⢀⠀⡀⡄⡄⠨⠳⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣭⠀⠀⠈⢳⣴⣧⡈⠈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢒⡀⠀⢐⡋⠺⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⠄⠀⠐⠀⠀⠈⠻⣿⣟⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠅⠀⠭⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣒⠀⡉⠀⠀⠀⠀⠀⠀⠈⠻⣿⣵⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡂⠰⣔⡂⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣟⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡛⢮⡉⠁⠃⠛⠘⠰⠀⠆⠆⠄⢠⢠⢈⡈⠁⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢘⢯⠙⠛⠳⠶⠄⣤⡉⎥
⎢⢰⠀⠹⢦⡀⠀⢀⢀⢀⠀⡄⡄⡄⠠⠰⠹⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡎⢧⠀⣀⣠⣤⠠⠿⎥
⎢⠬⠀⠀⠀⢙⣶⣫⣈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⠅⠈⣿⡍⠀⠀⠀⠀⎥
⎢⢘⡃⠀⢀⡉⠻⣿⣿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡃⢈⢹⣿⡄⠀⠀⠀⎥
⎢⠀⠂⠀⢐⠂⠀⠈⠻⣿⢿⣶⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠂⢐⠀⢻⣿⡆⠀⠀⎥
⎢⠀⠥⠀⠠⠀⠀⠀⠀⠀⠻⣿⣿⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠥⠠⠀⠀⢻⣿⣄⠀⎥
⎢⠀⢉⠀⠭⠀⠀⠀⠀⠀⠀⠈⠻⢟⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢉⠭⠀⠀⠀⠻⣿⡄⎥
⎢⣂⢐⣦⡂⠀⣀⠀⠀⠀⠀⠀⠀⠈⠻⣿⢿⡃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣰⡆⣀⠀⠀⠀⢻⣿⎥
⎢⢫⠛⠾⣥⣁⠉⣈⢈⢀⠁⡁⡃⡜⢰⠰⠮⠅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡿⣯⣉⣉⣉⣣⢳⠮⎥
⎢⢻⡄⠀⢀⡿⠿⣷⣶⣤⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⢸⢻⣶⣄⠀⠀⠀⎥
⎢⠀⠧⠀⣸⠁⠀⠀⠉⠛⠿⣿⣶⣤⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠧⣸⠀⠙⢿⣷⣄⠀⎥
⎣⢄⠹⣤⡖⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣿⣶⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⡹⡖⠀⠀⠀⠙⢿⣷⎦

In [10]:
fv = ∫(Pv ⋅ [100, 0], Γ="right")
fT = ∫(PT ⋅ 0, Γ="right") #500

nodal ScalarField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [11]:
T0 = 0
fv_T0 = ∫(SymGrad(Pv) ⋅ D ⋅ αI ⋅ T0)

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [12]:
fu = ∫(Pu ⋅ [0, 0])

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [13]:
F = SystemVector([fu, fv + fv_T0, fT])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :fu), Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)], [0, 242, 484])

In [14]:
M = SystemMatrix([Mu Muv MuT; Mvu Mv MvT; MTu MTv MT])

M = consistentToLumped(M)

M[:, :]

605×605 SparseArrays.SparseMatrixCSC{Float64, Int64} with 605 stored entries:
⎡⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⎦

In [15]:
supp1 = BoundaryCondition("left", problem=Pu, ux=0)
supp2 = BoundaryCondition("bottom", problem=Pu, uy=0)

supp3 = BoundaryCondition("left", problem=Pv, vx=0)
supp4 = BoundaryCondition("bottom", problem=Pv, vy=0)

suppT = BoundaryCondition("left", problem=PT, T=0)

BoundaryCondition("left", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q), Dict{Symbol, Union{Function, Number, ScalarField}}(:T => 0))

In [16]:
X0 = SystemVector([0fu, 0fv, 0fT])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :fu), Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)], [0, 242, 484])

In [17]:
u, v, T = FDM(K, M, F, X0, 10, 0.0000001, support=[supp1, supp2, suppT])

(VectorField(Matrix{Float64}[], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 -0.00019482287261573066 … -1.603876072038632e13 -4.457100790878545e15; 0.0 -3.909114538468107e-5 … 1.167827099413176e13 3.2403821451069105e15], [0.0, 1.0e-7, 2.0e-7, 3.0e-7, 4.0e-7, 5.0e-7, 6.0e-7, 7.0e-7, 8.0e-7, 9.0e-7], Int64[], 10, :v2D, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :fu)), VectorField(Matrix{Float64}[], [0.0 22619.754140216286 … 4.469314424202876e20 1.2406039596355866e23; 0.0 41868.9771958994 … -1.1955205231355846e21 -3.327378370784441e23; … ; 0.0 22674.82642344787 … 2.0329565277023767e21 5.652058812424947e23; 0.0 1410.5426635518995 … -1.9888376756659502e21 -5.513522080023687e23], [0.0, 1.0e-7, 2.0e-7, 3.0e-7, 4.0e-7, 5.0e-7, 6.0e-7, 7.0e-7, 8.0e-

In [18]:
showDoFResults(u, name="u", factor=0.000001, visible=true)
showDoFResults(v, name="v", factor=0.000001)
showDoFResults(T, name="T")

2

In [19]:
showDoFResults(T, name="T")

3

In [20]:
εx = ∂x(u[1])
εy = ∂y(u[2])
γxy = ∂y(u[1]) + ∂x(u[2])

ε = [εx, εy, γxy]

3-element Vector{ScalarField}:
 ScalarField([[0.0 -0.0011030099795077998 … 1.507906561750783e14 4.198868357052366e16; 0.0 -0.0011030099795077996 … 1.507906561750784e14 4.198868357052369e16; 0.0 -0.002294391674014994 … -1.1968524059131958e14 -3.3275545129446132e16; 0.0 -0.0022943916740149947 … -1.1968524059131967e14 -3.3275545129446156e16], [0.0 -0.0022943916740149947 … -1.1968524059131967e14 -3.3275545129446156e16; 0.0 -0.0022943916740149947 … -1.1968524059131967e14 -3.3275545129446156e16; 0.0 -0.0009733739172182923 … 2.2643395537284785e13 6.275354148318199e15; 0.0 -0.0009733739172182923 … 2.2643395537284785e13 6.275354148318199e15], [0.0 -0.0009733739172182923 … 2.2643395537284785e13 6.275354148318199e15; 0.0 -0.0009733739172182908 … 2.26433955372848e13 6.275354148318202e15; 0.0 0.00037829212077329335 … 3.3516116994968168e13 9.339134510775004e15; 0.0 0.0003782921207732919 … 3.3516116994968152e13 9.339134510775e15], [0.0 0.0003782921207732919 … 3.3516116994968152e13 9.339134510775e15; 

In [21]:
T = nodesToElements(T)
εth = [α * T, α * T, 0]

3-element Vector{Any}:
  ScalarField([[0.0 0.0 … 0.0 0.0; 0.0 0.006729637753096074 … -2.0162538091100022e14 -5.627384803980771e16; 0.0 -0.003997403090911181 … 2.178148949319192e13 6.140201661303344e15; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -0.003997403090911181 … 2.178148949319192e13 6.140201661303344e15; 0.0 0.0028295382553528425 … 2.1668908525986447e14 6.025407586954791e16; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0028295382553528425 … 2.1668908525986447e14 6.025407586954791e16; 0.0 -9.516719071747483e-5 … -4.750861044636315e14 -1.321396818765593e17; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -9.516719071747483e-5 … -4.750861044636315e14 -1.321396818765593e17; 0.0 0.0007515105804500531 … 6.421378178112389e14 1.7867135630182576e17; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0007515105804500531 … 6.421378178112389e14 1.7867135630182576e17; 0.0 0.00222157351845414 … -5.366673386863346e14 -1.4931778085051712e17; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.002221573518454

In [22]:
σ = D * (ε - εth)

3-element Vector{ScalarField}:
 ScalarField([[0.0 -94.44144701554852 … 2.111337247816449e19 5.879331330663774e21; 0.0 -1705.0139091729625 … 8.754747319941525e19 2.4409185785951174e22; 0.0 635.1276326072643 … -1.5370843918354985e19 -4.288519408803721e21; 0.0 -313.0435927966852 … -2.8515232433101193e19 -7.930618889514174e21], [0.0 -323.3909333377801 … -5.846008666646694e18 -1.6190791582965794e21; 0.0 604.7596071034789 … -5.186004200087729e19 -1.444830351315016e22; 0.0 -781.2965384646914 … -7.223641601336718e19 -2.009915437424256e22; 0.0 -81.00235410902741 … 2.0269337411996324e19 5.637966580742751e21], [0.0 -174.59518534594673 … -1.2500117711833836e19 -3.485353350411403e21; 0.0 -880.9236342556918 … -2.211365224694153e19 -6.145076872144994e21; 0.0 64.72464425494975 … 1.4489193822979752e20 4.030917366590933e22; 0.0 73.41693171672928 … -1.0505122948956154e19 -2.92319181601566e21], [0.0 25.800871835618615 … 1.621443489943089e19 4.51525154825459e21; 0.0 102.06676316394915 … 1.0206850800579718e

In [23]:
σx = σ[1]
σy = σ[2]
τxy = σ[3]

elementwise ScalarField
[[0.0 0.0 … 0.0 0.0; 0.0 -76.51075102339787 … -1.737001171894299e19 -4.833482577062282e21; 0.0 -82.73814790814049 … 4.0266236396384584e18 1.1243746188693348e21; 0.0 -6.227396884742571 … 2.1396635358581457e19 5.95785719593162e21], [0.0 -6.227396884742571 … 2.1396635358581457e19 5.95785719593162e21; 0.0 78.60860584532088 … 3.0537006486106518e19 8.497823204595385e21; 0.0 49.023743170772974 … -1.7084516979118285e19 -4.760864074297693e21; 0.0 -35.81225955929048 … -2.622488810664334e19 -7.300830082961459e21], [0.0 -35.81225955929048 … -2.622488810664334e19 -7.300830082961459e21; 0.0 50.99198141264617 … -2.552663993963612e19 -7.104073545922941e21; 0.0 14.367143425003228 … 2.3559425300437463e19 6.560869513977639e21; 0.0 -72.43709754693332 … 2.286117713343024e19 6.364112976939121e21], [0.0 -72.43709754693332 … 2.286117713343024e19 6.364112976939121e21; 0.0 -65.95861426900993 … 1.4286202431421516e19 3.974919646364083e21; 0.0 -3.4655770015238785 … -1.7761218412956924e19 -4

In [24]:
showElementResults(σx, name="σx")
showElementResults(σy, name="σy")
showElementResults(τxy, name="τxy")

6

In [25]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
